# Notebook 00 — LangChain in 15 Minutes

**Workshop:** Agentic AI for Scientists — Week 2
**Block:** warm-up (15 min)
**Goal:** Learn the five LangChain pieces every later notebook leans on — *model wrapper, messages, prompt templates, output parsers, and the LCEL pipe* — plus a first look at **tools**. Nothing here is an "agent" yet. This is the vocabulary.

**The one-sentence version of LangChain:** it is a thin, vendor-neutral standard layer over "call an LLM, give it a prompt, maybe give it tools, parse what comes back." You can do all of it with raw SDK calls (we will, in Notebook 01). LangChain's value is that the *same code* works across Anthropic, OpenAI, Google, etc., and that chains/agents/retrievers snap together with one operator: `|`.


## 0. Setup

Install the two packages we need for the intro, then load your API key.

In [ ]:
%pip install -q "langchain==0.3.7" "langchain-google-genai==2.0.11" python-dotenv


In [ ]:
import os

# Free Gemini API key (no credit card): https://aistudio.google.com/apikey
# Colab: add GOOGLE_API_KEY under the key icon (left sidebar) -> "Secrets".
# Local: put GOOGLE_API_KEY=AIza... in a .env file next to this notebook.
try:
    from google.colab import userdata  # type: ignore
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY first (see the comment above)."
print("API key loaded.")


---

## 1. The model wrapper

`ChatGoogleGenerativeAI` is a *wrapper* around the Gemini API. The point of the wrapper: every chat model in LangChain — Gemini, Anthropic, OpenAI — exposes the **same** `.invoke()` method. Swap the class, keep the code.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

response = llm.invoke("In one sentence, what is an AI agent?")
print(type(response).__name__)   # AIMessage
print(response.content)


`llm.invoke(...)` returns an **`AIMessage`** object, not a bare string. The text is in `.content`; token counts and stop reasons live in `.response_metadata`. That object-not-string detail matters once we start chaining.

---

## 2. Messages

A chat model's real input is a *list of messages*, each with a role. LangChain gives you typed message classes. A `SystemMessage` sets behaviour; `HumanMessage` is the user turn; `AIMessage` is the model's reply (you append these to build multi-turn history).


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a terse research assistant. Answer in <= 12 words."),
    HumanMessage(content="What is retrieval-augmented generation?"),
]
print(llm.invoke(messages).content)


---

## 3. Prompt templates

Hard-coding prompts is fine until you need to reuse one with different inputs. `ChatPromptTemplate` is a prompt with `{placeholders}` you fill at call time.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {field}. Explain for a curious non-expert."),
    ("human", "Explain {concept} in 2 sentences."),
])

# .invoke fills the template and returns a list of messages
filled = prompt.invoke({"field": "machine learning", "concept": "embeddings"})
for m in filled.messages:
    print(f"[{m.type}] {m.content}")


---

## 4. Output parsers + the LCEL pipe

The big idea. LangChain Expression Language (**LCEL**) lets you connect components with `|`, exactly like a Unix pipe. Data flows left to right:

```
prompt  |  llm  |  parser
 dict   ->  messages -> AIMessage -> str
```

`StrOutputParser` just pulls `.content` out of the `AIMessage` so the chain returns a plain string.


In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"field": "biology", "concept": "a transformer model"})
print(answer)


That three-component chain is the spine of everything in LangChain. A RAG chain (Notebook 03) is the same pipe with a retriever bolted on the front. An agent (Notebooks 01–02) is the same pipe wrapped in a loop.

LCEL chains also come with `.batch()` and `.stream()` for free:


In [ ]:
# Run several inputs in one call
for out in chain.batch([
    {"field": "physics", "concept": "entropy"},
    {"field": "chemistry", "concept": "a catalyst"},
]):
    print("-", out, "\n")


---

## 5. Tools — a first look

A **tool** is a plain Python function the model is allowed to call. The `@tool` decorator wraps a function so LangChain can read its name, its docstring (the description the model sees), and its typed arguments (the schema the model fills in).


In [ ]:
from langchain_core.tools import tool

@tool
def word_count(text: str) -> int:
    """Count the number of words in a piece of text."""
    return len(text.split())

# The decorator turned the function into a Tool with metadata the model reads:
print("name:       ", word_count.name)
print("description:", word_count.description)
print("args schema:", word_count.args)

# You can still call it like a normal function:
print("result:     ", word_count.invoke({"text": "agents are loops around a model"}))


Three things the model needs are now machine-readable: **what the tool is called**, **what it does** (docstring), and **what arguments it takes** (type hints). That is the entire contract behind "function calling".

---

## 6. Where this is going

You now have the whole toolbox:

| Piece | Class | One-liner |
|---|---|---|
| Model wrapper | `ChatGoogleGenerativeAI` | vendor-neutral `.invoke()` |
| Messages | `SystemMessage` / `HumanMessage` / `AIMessage` | typed chat turns |
| Prompt template | `ChatPromptTemplate` | a prompt with `{slots}` |
| Output parser | `StrOutputParser` | `AIMessage` -> `str` |
| Chain | `prompt | llm | parser` | the LCEL pipe |
| Tool | `@tool` | a function the model can call |

An **agent** is just: *give the model some tools, let it pick one, run it, feed the result back, and loop until it's done.* That loop is the only new idea in the next two notebooks — and we build it by hand before letting LangChain hide it.

**Next:** [Notebook 01 — Tool Use & Function Calling](01_tool_use.ipynb)
